# SV2 tuần 11 · Độ bền cache model KV260

Xem một lượt đã lưu. Chạy từng ô bằng **Shift+Enter** theo thứ tự.
Số liệu tính theo KiB và ms. Lượt 100 lần dùng cho demo.
Xem trạng thái thực thi và nhận xét bộ nhớ riêng trước khi kết luận nghiệm thu.

In [ ]:
from pathlib import Path
from hashlib import sha256
from collections import Counter
import csv
import json

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'edge/server/week11/stress.py').is_file())
local = root / 'edge/build_vai25/ket_qua_hien_tai/week11'
pointer = local / 'analysis_path.txt'
analysis = root / 'results/week11/edge'
if pointer.is_file():
    analysis = Path(pointer.read_text().strip()).resolve()
    if not analysis.is_relative_to((local / 'analyses').resolve()):
        raise ValueError('Pointer nằm ngoài thư mục phân tích tuần 11')
report, memory, switches = None, [], []
manifest = analysis / ('sha256sum_results.txt' if pointer.is_file() else 'sha256sum_export.txt')
if not manifest.is_file():
    print('PENDING · Chưa có export. Nhận và phân tích lượt board tại D.1.')
    print(analysis)
else:
    checked = set()
    for line in manifest.read_text().splitlines():
        expected, name = line.split('  ', 1)
        if name in checked or Path(name).name != name:
            raise ValueError(f'Tên file inventory không hợp lệ {name}')
        if sha256((analysis / name).read_bytes()).hexdigest() != expected:
            raise ValueError(f'SHA mismatch {name}')
        checked.add(name)
    if not {'summary.json', 'memory.csv'} <= checked:
        raise ValueError('Thiếu summary/memory trong danh mục SHA')
    report = json.loads((analysis / 'summary.json').read_text())
    with (analysis / 'memory.csv').open(newline='') as f:
        memory = list(csv.DictReader(f))
    if 'switches.csv' in checked:
        with (analysis / 'switches.csv').open(newline='') as f:
            switches = list(csv.DictReader(f))
    print('Nguồn', analysis)
    print('SHA inventory raw', report['source_inventory_sha256'])
    print('Run', report['run_id'], '· seed', report['seed'], '· SHA kiểm đủ', len(checked))
    print('Lượt', 'đầy đủ' if report['full_workload'] else 'demo')
    print('Thực thi hoàn tất', report['execution_complete'], '· nghiệm thu', report['acceptance'])
    print('Chuyển split', report['validated_switches'], '/', report['planned_switches'],
          '· cặp có hướng', report['directed_pairs'], '/', report['max_directed_pairs'])
    print('Request', report['request_rows'], '· PID', report['worker_pid'],
          '· mã thoát', report['worker_returncode'], '· mismatch', report['parity_mismatches'])
    print('Mẫu RAM', report['memory_samples'], '· đủ chuỗi', report['memory_samples_complete'])
    if report['error']:
        print('Lỗi lượt chạy', report['error'])

In [ ]:
from html import escape
from IPython.display import display, HTML

def show_table(headers, rows):
    header = ''.join(f'<th>{escape(str(h))}</th>' for h in headers)
    body = ''.join('<tr>' + ''.join(f'<td>{escape(str(x))}</td>' for x in row)
                   + '</tr>' for row in rows)
    display(HTML(f'<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>'))

if report:
    print("Nguồn RSS", report.get("rss_source"), "· độ dốc KB/1.000 lần", report.get("rss_slope_KB_per_1000"))
    values = report['memory_stats']
    show_table(['Bộ nhớ (KiB)', 'Sau warm-up', 'Cuối', 'Min', 'Max', 'Cuối − đầu'],
               [(name, *(row[k] for k in ('start', 'end', 'min', 'max', 'end_minus_start')))
                for name, row in values.items()])
    windows = report['quarter_medians']
    show_table(['Đoạn stress', 'Mẫu', 'Trung vị RSS worker (KiB)', 'Trung vị RSS Python (KiB)'],
               [(f"{int(r['from_exclusive']) + 1}–{int(r['to_inclusive'])}", r['sample_count'],
                 r['worker_rss_median_kib'], r['driver_rss_median_kib']) for r in windows])
    show_table(['Thời gian (ms)', 'Mẫu', 'Mean', 'p95', 'Max'],
               [(name, row['count'], *[None if row[k] is None else round(row[k], 3)
                 for k in ('mean_ms', 'p95_ms', 'max_ms')])
                for name, row in [('SELECT có IPC', report['switch']),
                                  ('Tail CPU/DPU', report['tail'])]])
    print('SELECT ≥ 50 ms', report['switch']['at_or_above_50_count'])

In [ ]:
import matplotlib.pyplot as plt

def series(rows, key):
    return [float(r[key]) / 1024 if r.get(key) else float('nan') for r in rows]

if memory:
    warm = [r for r in memory if r['phase'] in ('after_warmup', 'random_switch')]
    x = [int(r['completed_switches']) for r in warm]
    fig, axes = plt.subplots(2, 2, figsize=(12, 7), layout='constrained')
    for axis, label, key in [(axes[0, 0], 'Worker GraphRunner', 'worker'),
                             (axes[0, 1], 'Python receiver', 'driver')]:
        for field, name in [('rss_kib', 'RSS'), ('pss_kib', 'PSS'),
                            ('private_kib', 'Private'), ('peak_rss_kib', 'RSS đỉnh')]:
            field = f'{key}_{field}'
            if any(r.get(field) for r in warm):
                axis.plot(x, series(warm, field), label=name, marker='.' if len(x) < 10 else None)
        axis.set(title=label, xlabel='Lần đổi split', ylabel='MiB')
        if axis.lines:
            axis.legend()
    for field, label in [('MemAvailable_kib', 'RAM khả dụng'), ('CmaFree_kib', 'CMA còn trống')]:
        if any(r.get(field) for r in warm):
            axes[1, 0].plot(x, series(warm, field), label=label)
    axes[1, 0].set(title='Bộ nhớ hệ thống', xlabel='Lần đổi split', ylabel='MiB')
    if axes[1, 0].lines:
        axes[1, 0].legend()
    initial = [r for r in memory if r['phase'] != 'random_switch']
    for field, label in [('worker_rss_kib', 'Worker'), ('driver_rss_kib', 'Python')]:
        axes[1, 1].plot([r['phase'] for r in initial], series(initial, field), 'o-', label=label)
    axes[1, 1].set(title='RSS lúc nạp và warm-up', ylabel='MiB')
    axes[1, 1].legend()
    for axis in axes.flat:
        axis.grid(alpha=.25)
    plt.show()
if switches:
    pairs = Counter((int(r['previous_s']), int(r['split_s'])) for r in switches)
    fig, axis = plt.subplots(figsize=(5, 4), layout='constrained')
    heat = axis.imshow([[pairs[i, j] for j in range(10)] for i in range(10)], cmap='Blues')
    axis.set(title='Số lần chuyển theo cặp split', xlabel='Split đến', ylabel='Split trước',
             xticks=list(range(10)), yticks=list(range(10)))
    fig.colorbar(heat, ax=axis, label='Lần')
    plt.show()

So sánh RSS và private mỗi tiến trình qua bốn đoạn bằng nhau sau warm-up.
Nếu RAM còn tăng đến cuối, ghi xu hướng và đối chiếu log worker/kernel
trước khi kết luận. Dựa vào số lần, mã thoát và run ID của đúng lượt khi demo.

## Echo và queue tải bùng
Ô này đọc và kiểm SHA của kết quả HOST và bảng thời gian chờ theo q.

In [ ]:
import sys

sys.path.insert(0, str(root / 'edge/server/week10'))
from evidence import verify

published = root / 'results/week11/edge'
for pointer_name, label, summary_name, csv_name, status_field in (
    ('echo_path.txt', 'Echo trên HOST', 'echo_summary.json', 'echo_checks.csv', 'status'),
    ('burst_analysis_path.txt', 'Queue tải bùng trên KV260', 'burst_summary.json', 'wait_by_q.csv', 'workload_pass'),
):
    source_pointer = local / pointer_name
    if source_pointer.is_file():
        folder = Path(source_pointer.read_text().strip()).resolve()
        if not folder.is_relative_to(local.resolve()):
            raise ValueError(f'Pointer nằm ngoài kết quả tuần 11: {folder}')
        summary_name, inventory = 'summary.json', 'sha256sum_results.txt'
    else:
        folder, inventory = published, 'sha256sum_export.txt'
        if not (folder / inventory).is_file():
            print(label, '· Chưa có hồ sơ')
            continue
    if not {summary_name, csv_name} <= verify(folder, inventory):
        raise ValueError(f'Thiếu mục đã niêm checksum: {folder}')
    row = json.loads((folder / summary_name).read_text())
    print(label, '·', folder, '·', row[status_field])
    if pointer_name == 'echo_path.txt':
        print('Vector SV1', row['sv1_vectors'], '· bổ sung', row['supplemental_vectors'],
              '· số phép kiểm ghép khung', row['binding_checks'])
        with (folder / 'echo_checks.csv').open(newline='') as stream:
            cases = list(csv.DictReader(stream))
        show_table(['Nguồn', 'Khung (B)', 'Mode', 'MTU (B)', 'Phản hồi (B)'],
                   [(case['source'], case['input_bytes'], case['mode'], case['mtu'],
                     case['response_bytes']) for case in cases])
    else:
        print({key: row[key] for key in ('q_max', 'arrived', 'completed', 'dropped', 'failed')})
        print('Độ dốc chờ (ms/job)', row['queue_wait_slope_ms_per_job'],
              '· Mốc tuần 10', row['week10_reference_ms_per_job'])
        with (folder / 'wait_by_q.csv').open(newline='') as stream:
            waits = list(csv.DictReader(stream))
        show_table(['q', 'Mẫu', 'Mean chờ (ms)', 'p95 chờ (ms)'],
                   [(r['q'], r['samples'], round(float(r['mean_wait_ms']), 3),
                     round(float(r['p95_wait_ms']), 3)) for r in waits])
        fig, ax = plt.subplots(figsize=(8, 4))
        depth = [int(r['q']) for r in waits]
        for key, name in (('mean_wait_ms', 'Mean'), ('p95_wait_ms', 'p95')):
            ax.plot(depth, [float(r[key]) for r in waits], marker='o', label=name)
        ax.set(xlabel='q sau enqueue', ylabel='Thời gian chờ (ms)')
        ax.legend()
        ax.grid(alpha=.3)
        plt.show()
